In [ ]:
import requests 
from selenium import webdriver 
from bs4 import BeautifulSoup as bs
import pandas as pd
import time
import requests

In [ ]:
base_url = "https://www.viagginaturaecultura.it"
viaggi_url_list =['https://www.viagginaturaecultura.it/search?view=list&destination=d110&page=1',
           'https://www.viagginaturaecultura.it/search?view=list&destination=d110&page=2'
           'https://www.viagginaturaecultura.it/search?view=list&destination=d110&page=3'
           'https://www.viagginaturaecultura.it/search?view=list&destination=d110&page=4'
           'https://www.viagginaturaecultura.it/search?view=list&destination=d110&page=5'
           'https://www.viagginaturaecultura.it/search?view=list&destination=d110&page=6',
          'https://www.viagginaturaecultura.it/search?view=list&destination=europe&page=1',
           'https://www.viagginaturaecultura.it/search?view=list&destination=europe&page=2',
           'https://www.viagginaturaecultura.it/search?view=list&destination=europe&page=3',
          'https://www.viagginaturaecultura.it/search?view=list&destination=europe&page=4',
          'https://www.viagginaturaecultura.it/search?view=list&destination=world&page=1',
           'https://www.viagginaturaecultura.it/search?view=list&destination=world&page=2',
           'https://www.viagginaturaecultura.it/search?view=list&destination=world&page=3',
          'https://www.viagginaturaecultura.it/search?view=list&destination=world&page=4']
driver = webdriver.Chrome()
list_trips = set()

for viaggio_url in viaggi_url_list:
    driver.get(viaggio_url)  
    time.sleep(5)  
    html = driver.page_source 
    soup = bs(html, "html.parser") 
    links = soup.find_all("a")
    for link in links:
        product_url = link.get("href")
        if product_url.startswith("/product/"):
            list_trips.add(base_url + product_url)
        
driver.close()

In [ ]:
list_trips

In [ ]:
df = pd.DataFrame(columns = ["Country", "Nome_Viaggio", "Link", "Descrizione Viaggio", "Mood Viaggio", "Itinerario", "Sottotitolo_Viaggio", "PDF_Link"])
for trip in list_trips:
    response = requests.get(trip)
    soup = bs(response.content, "html.parser")
    header = soup.find(attrs = {"class": "col-sm-8"})
    title = header.find('h1').text
    subtitle = header.find('p').text
    
    tag_list = []
    tags = soup.find_all(attrs = {"class": "product-tag__tag"})
    for tag in tags:
        tag_list.append(tag.text.strip())
    properties = soup.find(attrs = {"class": "row sidebar--product__meta"})
    durata = properties.find("span").text
    difficolta = len(properties.find_all(attrs = {"class": "fas fa-paw"}))
    
    
    description_section = soup.find(attrs = {"class": "col-sm-7"})
    pdf_link = description_section.find_all('a')[-1].get('href')
    
    description_text = description_section.find_all('p')[0].text.strip()
    try:
        come_raggiungere = description_section.text.split("COME SI RAGGIUNGE")[1].split("\n\n\n")[0].strip()
    except:
        come_raggiungere = ""
        
    mood_viaggio = {"Tags": tag_list,
                   "Durata": durata,
                   "Difficoltà": difficolta,
                   "ComeRaggiungere": come_raggiungere}
    
    list_days_itinerario = soup.find_all(attrs = {"class": "itinerary-day has-shadow"})
    itinerario = {}
    for day in list_days_itinerario:
        giorno = day.find("h6").text.strip()
        titolo_giorno = day.find("h5").text.strip()
        descr_giorno = day.find("p").text.strip()
        itinerario[giorno] = {"Titolo": titolo_giorno,
                             "Descrizione": descr_giorno}
    
    row = pd.DataFrame.from_dict({"Country": [""],
                         "Nome_Viaggio": [title],
                          "Link": [trip],
                          "Descrizione Viaggio": [description_text],
                          "Mood Viaggio": [mood_viaggio],
                         "Itinerario": [itinerario],
                         "Sottotitolo_Viaggio": [subtitle],
                         "PDF_Link": [pdf_link]})
    df = pd.concat([df, row], ignore_index=True)
    

In [ ]:
df

In [ ]:
df.to_pickle("df_viaggi_fourseasons.pkl")

In [ ]:
list_days_itinerario = soup.find_all(attrs = {"class": "itinerary-day has-shadow"})
list_days_itinerario[0].find("p").text.strip()

In [ ]:
soup

In [ ]:
properties = soup.find(attrs = {"class": "row sidebar--product__meta"})

In [ ]:
properties.find("span").text

In [ ]:
len(soup.find_all(attrs = {"class": "fas fa-paw"}))

In [ ]:
soup

In [ ]:
cards

In [ ]:
df = pd.DataFrame(columns = ["Country", "Nome_Viaggio", "Link", "Descrizione Viaggio", "Mood Viaggio", "Itinerario"])

url = "https://www.sivola.it"
downloaded = trafilatura.fetch_url(url + "/viaggi")
soup = bs(downloaded, 'lxml')
l = soup.find_all(attrs={"class": "card-viaggio rounded"})
set_urls = set()
for i, el in enumerate(l):
    print(i)
    link_element = el.find(attrs={"class": "font-weight-bolder h4"}).find('a')
    nome_viaggio = link_element.text.strip()
    url_viaggio = url + link_element.get('href')
    if url_viaggio in set_urls:
        continue
    else:
        set_urls.add(url_viaggio)
    
    downloaded_viaggio = trafilatura.fetch_url(url_viaggio)
    soup_viaggio = bs(downloaded_viaggio, 'lxml')
    
    descr_viaggio = soup_viaggio.find(attrs = {"class": "h3 viaggio--title"}).text.strip()
    prose_viaggio = soup_viaggio.find(attrs = {"class": "prose"}).text.strip()
    if "_________________________________________________________________" in prose_viaggio:
        prose_viaggio = prose_viaggio.split("_________________________________________________________________")[1].strip()
    
    descr_viaggio = descr_viaggio + "\n" + prose_viaggio
    
    card_elements = soup_viaggio.find_all(attrs = {"class": "card-ranking"})
    mood_viaggio = {}
    for card in card_elements:
        card_type = card.find("small").text
        card_score = card.find("p").text
        mood_viaggio[card_type] = card_score
    
    itinerario_dict = {}
    itinerario = soup_viaggio.find(attrs = {"class": "row mt-5 mb-5"})
    tappe_itinerario = itinerario.find_all(attrs = {"class": "card-home card-tappa rounded p-4"})
    for day, tappa in enumerate(tappe_itinerario):
        itinerario_dict[f"Giorno {day+1}"] = {}
        titolo_giorno = tappa.find(attrs = {"class": "h4 font-weight-bolder pr-1"}).text.strip()
        sottotitolo_giorno = tappa.find(attrs = {"class": "mb-0 color-h text-small"}).text.strip()
        testo_giorno = tappa.find(attrs = {"class": "mt-4"}).text.strip()
        itinerario_dict[f"Giorno {day+1}"]['Title'] = titolo_giorno
        itinerario_dict[f"Giorno {day+1}"]['Subtitle'] = sottotitolo_giorno
        itinerario_dict[f"Giorno {day+1}"]['Tappe'] = testo_giorno
        
    row = pd.DataFrame.from_dict({"Country": [nome_viaggio.split(":")[0].strip()],
                             "Nome_Viaggio": [nome_viaggio],
                              "Link": [url_viaggio],
                              "Descrizione Viaggio": [descr_viaggio],
                              "Mood Viaggio": [mood_viaggio],
                             "Itinerario": [itinerario_dict]})
    df = pd.concat([df, row], ignore_index=True)
    print("*"*80)

In [ ]:
df

In [ ]:
df.to_pickle("df_viaggi_sivola.pkl")